In [ ]:
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)


In [ ]:
# Import libraries here

import sqlite3
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from category_encoders import OrdinalEncoder
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from category_encoders import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree


In [ ]:
%load_ext sql
%sql sqlite:///../nepal.sqlite


In [ ]:
# Build your `wrangle` function here
def wrangle(db_path):
    # Connect to database
    conn = sqlite3.connect(db_path)

    # Construct query
    query = """
        SELECT distinct(i.building_id) as b_id,
           b.*,
           d.damage_grade
        from id_map as i
        JOIN building_structure as b ON i.building_id = b.building_id
        JOIN building_damage as d ON i.building_id = d.building_id
        WHERE i.district_id = 3
    """
    # Read query results into DataFrame
    df = pd.read_sql(query, conn, index_col="b_id")

    # Create binary target
    df['damage_grade'] = df['damage_grade'].str[-1].astype(int)
    df['severe_damage'] = (df['damage_grade']>3).astype(int)

    # Identify leaky columns
    drop_cols = [col for col in df.columns if "post_eq" in col]

    drop_cols.extend(("damage_grade", "count_floors_pre_eq", "building_id"))

    # Drop columns
    df.drop(columns=drop_cols, inplace=True)

           
    return df


In [ ]:
df = wrangle("../nepal.sqlite")
df.head()


In [ ]:
fig, ax = plt.subplots() 

# Calculate value counts and plot on the axes object
df["severe_damage"].value_counts(normalize=True).plot(
    kind= "bar",
    ax=ax  # Direct the plot to our Axes object
)

# Set labels and title 
ax.set_xlabel("Severe Damage")
ax.set_ylabel("Relative Frequency")
ax.set_title("Kavrepalanchok, Class Balance");


In [ ]:
fig, ax = plt.subplots() 

# Create the Seaborn boxplot 
sns.boxplot(x="severe_damage", y="plinth_area_sq_ft", data=df, ax=ax)

# Set labels and title
ax.set_xlabel("Severe Damage")
ax.set_ylabel("Plinth Area [sq. ft.]")
ax.set_title("Kavrepalanchok, Plinth Area vs Building Damage");


In [ ]:
roof_pivot = pd.pivot_table(df, index="roof_type", values="severe_damage", aggfunc =np.mean).sort_values("severe_damage", ascending=True)
roof_pivot


In [ ]:
target = "severe_damage"
X = df.drop(columns=target)
y = df[target]
print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)


In [ ]:
acc_baseline = y_train.value_counts(normalize=True).max()
print("Baseline Accuracy:", round(acc_baseline, 2))


In [ ]:
model_lr = make_pipeline(
    OneHotEncoder(use_cat_names=True),
    LogisticRegression(max_iter=2000)
)
# Fit model to training data
model_lr.fit(X_train, y_train)


In [ ]:
lr_train_acc = accuracy_score(y_train, model_lr.predict(X_train) )
lr_val_acc = model_lr.score(X_val, y_val)

print("Logistic Regression, Training Accuracy Score:", lr_train_acc)
print("Logistic Regression, Validation Accuracy Score:", lr_val_acc)


In [ ]:
depth_hyperparams = range(1, 16)
training_acc = []
validation_acc = []
for d in depth_hyperparams:
    model_dt = make_pipeline(
        OrdinalEncoder(),
        DecisionTreeClassifier(max_depth=d, random_state=42)
    )
    model_dt.fit(X_train, y_train)
    
    # Calculate training accuracy score and append to `training_acc`
    training_acc.append(model_dt.score(X_train, y_train))
    
    # Calculate validation accuracy score and append to `training_acc`
    validation_acc.append(model_dt.score(X_val, y_val))


In [ ]:
# Run this cell
submission = pd.Series(validation_acc, index=depth_hyperparams)
submission


In [ ]:
fig, ax = plt.subplots() 

#  Plot the training accuracy
ax.plot(depth_hyperparams, training_acc, label="training")

#  Plot the validation accuracy 
ax.plot(depth_hyperparams, validation_acc, label="validation")

#  Set labels and title  
ax.set_xlabel("Max Depth")
ax.set_ylabel("Accuracy Score")
ax.set_title("Validation Curve, Decision Tree Model")

# Add the legend 
ax.legend()


In [ ]:
final_model_dt = make_pipeline(
        OrdinalEncoder(),
        DecisionTreeClassifier(max_depth=10, random_state=42)
    )

final_model_dt.fit(X, y)


In [ ]:
X_test = pd.read_csv("data/kavrepalanchok-test-features.csv", index_col="b_id")
X_test = X_test[X_train.columns]
y_test_pred = final_model_dt.predict(X_test)
y_test_pred[:5]


In [ ]:
features = X_train.columns
importances = final_model_dt.named_steps["decisiontreeclassifier"].feature_importances_

feat_imp = pd.Series(importances, index=features).sort_values()
feat_imp.head()


In [ ]:
fig, ax = plt.subplots() 

# Create the horizontal bar plot on the axes object
feat_imp.plot(kind="barh", ax=ax)

# Set labels and title 
ax.set_xlabel("Gini Importance")
ax.set_ylabel("Feature")
ax.set_title("Kavrepalanchok Decision Tree, Feature Importance")

# Apply tight layout 
fig.tight_layout()
